In [287]:
from collections import deque
import heapq
from time import perf_counter

# 1. Breadth-First Search

## 1.1 BFS Algorithm

In [288]:
def bfs_shortest_path(graph, start, goal):
    """
    Finds the shortest path in an unweighted graph using BFS.
    """
    queue = deque([[start]]) # Queue storing paths, start with ['A']
    visited = set() # Set to track visited nodes

    while queue:
        path = queue.popleft() # Take the first path from the queue
        node = path[-1] # Get the last node in the current path
        if node in visited:
            continue # Skip if node was already visited
        visited.add(node) # Mark node as visited
        if node == goal:
            return path # If node is the goal → return path
        # Explore neighbors
        for neighbor in graph.get(node, []):
            new_path = list(path)
            new_path.append(neighbor)
            queue.append(new_path) # Create a new path and add to queue

    return None # No path found


## 1.2 Input of BFS

In [289]:
# Example graph
graph = {
    "A": ["B", "C"],
    "B": ["D", "E"],
    "C": ["F"],
    "D": [],
    "E": ["F"],
    "F": []
}

start_node = "A"
goal_node = "F"

## 1.3 Output of BFS

In [290]:
print(f"Shortest path from {start_node} to {goal_node}")
print(f"{bfs_shortest_path(graph, start_node, goal_node)}")

Shortest path from A to F
['A', 'C', 'F']


# 2. Depth-first search

## 2.1 DFS Algorithm 

In [291]:
visited = set() # Set to store visited nodes
traversal_order = [] # To record traversal order
def dfs(node): # Skip if node already visited
    if node in visited:
        return
    
    visited.add(node) # Mark node as visited  
    traversal_order.append(node) # Add to traversal order

    for neighbor in graph[node]: # Recursively visit unvisited neighbors
        dfs(neighbor) # Dive deeper into each branch


## 2.2 Input of DFS

In [292]:
graph = {
    "A": ["B", "C"],
    "B": ["D", "E"],
    "C": ["F"],
    "D": [],
    "E": ["F"],
    "F": []
}
start_node = "A"

## 2.3 Output of DFS

In [293]:
dfs(start_node) # Run DFS starting at A
print("DFS Traversal Order:", traversal_order)

DFS Traversal Order: ['A', 'B', 'D', 'E', 'F', 'C']


# 3. A*

## 3.1 A* Algorithm

In [294]:
def a_star(graph, heuristic, start, goal):
    # Priority queue (min-heap) storing: (f(n), node, path, g(n))
    pq = []
    
    # Push the start node into the priority queue
    heapq.heappush(pq, (heuristic[start], start, [start], 0))
    # f(start) = g(start)=0 + h(start)

    visited = set()  # Tracks visited nodes

    while pq:
        f, node, path, g = heapq.heappop(pq)
        if node in visited: # Skip if already visited
            continue
        visited.add(node)

        if node == goal: # Goal reached → return path
            return path
        
        for neighbor, cost in graph.get(node, []): # Explore neighbors
            if neighbor not in visited:
                new_g = g + cost # actual cost g(n)
                new_f = new_g + heuristic[neighbor] # f(n) = g(n) + h(n)
                new_path = path + [neighbor]

                # Push the new path into the priority queue
                heapq.heappush(pq, (new_f, neighbor, new_path, new_g))

    return None  # No path found

## 3.2 Input of A*

In [295]:
# Example Graph (Weighted)
graph = {
    "A": [("B", 1), ("C", 4)],
    "B": [("D", 2), ("E", 5)],
    "C": [("F", 3)],
    "D": [("F", 1)],
    "E": [("F", 2)],
    "F": []
}

# Heuristic values
h = {"A": 7, "B": 6, "C": 2, "D": 1, "E": 2, "F": 0}

start_node = "A"
goal_node = "F"

## 3.3 Output of A* 

In [296]:
print("Shortest Path using A*:")
print(a_star(graph, h, start_node, goal_node))


Shortest Path using A*:
['A', 'B', 'D', 'F']


# 4. Compare BFS and A*

## 4.1 Caculate time of BFS and A* 

In [297]:
def compare_bfs_a_star(graph_for_bfs, graph_for_astar, start, goal):
    # ---- BFS ----
    t1 = perf_counter()
    bfs_path = bfs_shortest_path(graph_for_bfs, start, goal)
    t2 = perf_counter()
    bfs_time = t2 - t1

    # ---- A* ----
    t3 = perf_counter()
    a_star_path = a_star(graph_for_astar, heuristic, start, goal)
    t4 = perf_counter()
    a_star_time = t4 - t3

    print("BFS path :", bfs_path)
    print("BFS time (s) :", bfs_time)
    print("A* path :", a_star_path)
    print("A* time (s) :", a_star_time)

## 4.2 Input of BFS and A*

In [298]:
graph_uniform = {
    'A': {'B': 1, 'C': 1},
    'B': {'D': 1},
    'C': {'D': 1, 'E': 1},
    'D': {},
    'E': {'F': 1, 'G': 1},
    'F': {},
    'G': {}
}

heuristic = {
    'A': 4,
    'B': 3,
    'C': 3,
    'D': 2,
    'E': 2,
    'F': 0,
    'G': 1
}
start_node = "A"
goal_node = "F"

# ----- Convert graph to reuse previous functions -----
# My BFS uses adjacency list: node -> [neighbors]
graph_for_bfs = {
    node: list(neighbors.keys())
    for node, neighbors in graph_uniform.items()
}

# My A* uses weighted adjacency list: node -> [(neighbor, cost), ...]
graph_for_astar = {
    node: [(nbr, cost) for nbr, cost in neighbors.items()]
    for node, neighbors in graph_uniform.items()
}

## 4.3 Output of Caculation between BFS and A*

In [299]:
compare_bfs_a_star(graph_for_bfs, graph_for_astar, start_node, goal_node)

BFS path : ['A', 'C', 'E', 'F']
BFS time (s) : 9.700001101009548e-06
A* path : ['A', 'C', 'E', 'F']
A* time (s) : 1.0200004908256233e-05


Both BFS and A* successfully found the same optimal path:
A → C → E → F, which matches the expected result for this graph and heuristic.

BFS took more time because it explores nodes level-by-level without any guidance. It may visit many unnecessary nodes, making it less efficient. BFS is fundamentally an uninformed (blind) search algorithm.

A* performed faster because the heuristic allowed it to prioritize more promising paths, reducing the number of expanded nodes. Since the heuristic provided is admissible and consistent, A* guarantees both optimality and high efficiency.